# CurvInspect Demo: Discrete Curvature and Boundary Inspection

This notebook demonstrates the main ideas behind **CurvInspect**:

- modeling an object boundary as a closed polygonal curve,
- computing discrete curvature,
- comparing curvature-based and chord-based inspection signals,
- detecting candidate boundary irregularities,
- visualizing final and debug overlays,
- reading the generated JSON report.

The notebook is designed to be run from the root of the repository.

## 1. Setup

The following cell makes the notebook robust whether the package is installed in editable mode or the notebook is run directly from the repository.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()

if (REPO_ROOT / "src").exists():
    sys.path.insert(0, str(REPO_ROOT / "src"))

from curvinspect.pipeline import analyze_image
from curvinspect.synthetic import save_demo_images

## 2. Helper Functions

These small helpers keep the notebook clean and make it easier to display images and reports.

In [ ]:
def show_image(path: str | Path, title: str | None = None, figsize: tuple[int, int] = (7, 7)) -> None:
    path = Path(path)
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)

    if image is None:
        raise FileNotFoundError(f"Could not read image: {path}")

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=figsize)
    plt.imshow(image_rgb)
    plt.axis("off")

    if title is not None:
        plt.title(title)

    plt.show()


def load_report(path: str | Path) -> dict:
    path = Path(path)
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def summarize_report(report: dict) -> pd.DataFrame:
    rows = [
        ("Input path", report.get("input", {}).get("path")),
        ("Image shape", report.get("input", {}).get("image_shape")),
        ("Contour area", report.get("contour", {}).get("area")),
        ("Contour perimeter", report.get("contour", {}).get("perimeter")),
        ("Analysis contour", report.get("contour", {}).get("analysis_contour")),
        ("Resampled points", report.get("contour", {}).get("num_resampled_points")),
        ("Inspection signal", report.get("inspection", {}).get("signal")),
        ("Curvature mode", report.get("inspection", {}).get("curvature_mode")),
        ("Deviation mode", report.get("inspection", {}).get("deviation_mode")),
        ("Chord mode", report.get("inspection", {}).get("chord_mode")),
        ("Chord step", report.get("inspection", {}).get("chord_step")),
        ("Anomaly regions", report.get("anomalies", {}).get("count")),
        ("Raw anomaly points", len(report.get("anomalies", {}).get("indices", []))),
        ("Max absolute curvature", report.get("curvature", {}).get("max_abs_curvature")),
        ("Mean absolute curvature", report.get("curvature", {}).get("mean_abs_curvature")),
        ("Bending energy", report.get("curvature", {}).get("bending_energy")),
        ("Total curvature", report.get("curvature", {}).get("total_curvature")),
    ]

    return pd.DataFrame(rows, columns=["Metric", "Value"])

## 3. Generate Synthetic Demo Images

CurvInspect includes a synthetic demo generator. It creates a normal reference shape and a defective sample.

In [ ]:
input_dir = REPO_ROOT / "examples" / "input"
output_dir = REPO_ROOT / "examples" / "output" / "notebook_demo"

input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

demo_images = save_demo_images(input_dir)

demo_images

In [ ]:
show_image(demo_images["normal"], "Synthetic normal part")
show_image(demo_images["defective"], "Synthetic defective part")

## 4. Analyze Normal and Defective Synthetic Parts

We now run the same inspection pipeline on both images. This gives a quick comparison between a reference-like boundary and a boundary with additional geometric irregularities.

In [ ]:
normal_result = analyze_image(
    input_path=demo_images["normal"],
    output_dir=output_dir / "normal",
    contour_method="threshold",
    analysis_contour="raw",
    inspection_signal="curvature",
    curvature_mode="absolute",
    num_samples=300,
    smoothing="none",
    anomaly_threshold=1.5,
)

defective_result = analyze_image(
    input_path=demo_images["defective"],
    output_dir=output_dir / "defective",
    contour_method="threshold",
    analysis_contour="raw",
    inspection_signal="curvature",
    curvature_mode="absolute",
    num_samples=300,
    smoothing="none",
    anomaly_threshold=1.5,
)

print("Normal anomaly regions:", len(normal_result.anomalies.peak_indices))
print("Defective anomaly regions:", len(defective_result.anomalies.peak_indices))

In [ ]:
show_image(normal_result.output_dir / "overlay.png", "Normal part: final overlay")
show_image(defective_result.output_dir / "overlay.png", "Defective part: final overlay")

In [ ]:
show_image(defective_result.output_dir / "overlay_debug.png", "Defective part: debug overlay")
show_image(defective_result.output_dir / "curvature_signal.png", "Defective part: inspection signal", figsize=(12, 5))

## 5. Create or Load a Real-Style Damaged Object

This cell creates a simple industrial-looking damaged object only if `examples/input/real/object_04.png` does not already exist.

If you already have your own `object_04.png`, the notebook will use it instead.

In [ ]:
real_input_dir = REPO_ROOT / "examples" / "input" / "real"
real_input_dir.mkdir(parents=True, exist_ok=True)

object_04_path = real_input_dir / "object_04.png"

if not object_04_path.exists():
    image = np.full((520, 520, 3), 245, dtype=np.uint8)

    part_points = np.array(
        [
            [115, 130],
            [380, 130],
            [380, 225],
            [330, 245],
            [380, 275],
            [380, 380],
            [115, 380],
            [115, 130],
        ],
        dtype=np.int32,
    )

    cv2.fillPoly(image, [part_points], (70, 70, 70))

    upper_left_chip = np.array(
        [
            [115, 130],
            [165, 130],
            [115, 185],
        ],
        dtype=np.int32,
    )

    lower_chip = np.array(
        [
            [205, 380],
            [270, 380],
            [235, 352],
        ],
        dtype=np.int32,
    )

    cv2.fillPoly(image, [upper_left_chip], (245, 245, 245))
    cv2.fillPoly(image, [lower_chip], (245, 245, 245))

    cv2.imwrite(str(object_04_path), image)

show_image(object_04_path, "Real-style damaged object")

## 6. Industrial Chord-Based Inspection

Some boundary defects are shallow and do not always produce the strongest curvature peaks. For this reason, the chord-based signal can be useful for local edge damage.

The settings below correspond to the tuned configuration used in the README example.

In [ ]:
industrial_output = REPO_ROOT / "examples" / "output" / "real" / "object_04_chord_tuned_1"

industrial_result = analyze_image(
    input_path=object_04_path,
    output_dir=industrial_output,
    contour_method="threshold",
    analysis_contour="raw",
    inspection_signal="chord",
    chord_mode="absolute",
    chord_step=25,
    num_samples=700,
    smoothing="median",
    smoothing_window=11,
    anomaly_threshold=1.9,
)

print("Detected anomaly regions:", len(industrial_result.anomalies.peak_indices))
print("Raw anomalous points:", len(industrial_result.anomalies.indices))

In [ ]:
show_image(industrial_output / "overlay.png", "Chord inspection: final overlay")
show_image(industrial_output / "overlay_debug.png", "Chord inspection: debug overlay")
show_image(industrial_output / "curvature_signal.png", "Chord inspection: signal plot", figsize=(12, 5))

## 7. Read and Summarize the JSON Report

CurvInspect exports a structured JSON report for every analysis run. This makes the pipeline useful not only for visualization, but also for downstream analysis and reproducible experiments.

In [ ]:
report = load_report(industrial_output / "anomaly_report.json")
summary = summarize_report(report)

display(summary)

In [ ]:
print(json.dumps(report.get("inspection", {}), indent=2))

## 8. Compare Curvature vs Chord Signal on the Same Object

This comparison shows why multiple geometric signals are useful. Curvature emphasizes strong changes in tangent direction. Chord deviation is often more sensitive to local edge departures over a chosen neighborhood scale.

In [ ]:
comparison_output = REPO_ROOT / "examples" / "output" / "notebook_demo" / "comparison"

curvature_result = analyze_image(
    input_path=object_04_path,
    output_dir=comparison_output / "curvature",
    contour_method="threshold",
    analysis_contour="raw",
    inspection_signal="curvature",
    curvature_mode="absolute",
    num_samples=700,
    smoothing="median",
    smoothing_window=11,
    anomaly_threshold=1.9,
)

chord_result = analyze_image(
    input_path=object_04_path,
    output_dir=comparison_output / "chord",
    contour_method="threshold",
    analysis_contour="raw",
    inspection_signal="chord",
    chord_mode="absolute",
    chord_step=25,
    num_samples=700,
    smoothing="median",
    smoothing_window=11,
    anomaly_threshold=1.9,
)

comparison = pd.DataFrame(
    [
        {
            "Signal": "curvature",
            "Anomaly regions": len(curvature_result.anomalies.peak_indices),
            "Raw anomalous points": len(curvature_result.anomalies.indices),
        },
        {
            "Signal": "chord",
            "Anomaly regions": len(chord_result.anomalies.peak_indices),
            "Raw anomalous points": len(chord_result.anomalies.indices),
        },
    ]
)

display(comparison)

In [ ]:
show_image(comparison_output / "curvature" / "overlay.png", "Curvature signal: final overlay")
show_image(comparison_output / "chord" / "overlay.png", "Chord signal: final overlay")

## 9. Copy README Assets

The following cell copies the main visual example into `docs/assets`, so the images can be shown directly in the GitHub README.

In [ ]:
assets_dir = REPO_ROOT / "docs" / "assets"
assets_dir.mkdir(parents=True, exist_ok=True)

asset_map = {
    object_04_path: assets_dir / "object_04_input.png",
    industrial_output / "overlay.png": assets_dir / "object_04_overlay.png",
    industrial_output / "overlay_debug.png": assets_dir / "object_04_overlay_debug.png",
    industrial_output / "curvature_signal.png": assets_dir / "object_04_signal.png",
}

for source, destination in asset_map.items():
    image = cv2.imread(str(source), cv2.IMREAD_COLOR)

    if image is None:
        raise FileNotFoundError(f"Could not read image: {source}")

    cv2.imwrite(str(destination), image)

sorted(path.name for path in assets_dir.iterdir())

## 10. Conclusion

This notebook demonstrates the main CurvInspect workflow:

- image boundary extraction,
- polygonal curve modeling,
- discrete curvature analysis,
- chord-based boundary inspection,
- robust anomaly candidate detection,
- visual and JSON reporting.

The key idea is that digital object boundaries can be studied as polygonal curves, allowing differential-geometric ideas to be applied to practical image-based shape inspection.